# Lab 2F: Evaluation in Python

**Time**: ~20 min  
**Environment**: Jupyter kernel in VS Code  

In this exercise you will evaluate a RAG pipeline using the LLM-as-judge pattern.

The lab follows the same structure as the C# version. Run each cell in order to complete the steps.

In [ ]:
%pip install azure-cosmos azure-identity openai python-dotenv numpy --quiet

## Step 0: Initialize Connection

Set up the Cosmos client connection and Azure OpenAI clients.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os

ENDPOINT = os.environ.get("COSMOS_ENDPOINT")
FOUNDRY_ENDPOINT = os.environ.get("FOUNDRY_ENDPOINT")
DB_NAME = "WorkshopData"
CONT_NAME = "Docs"
COMPLETIONS_MODEL = os.environ.get("COMPLETIONS_MODEL", "phi-4-mini-instruct")
EVAL_MODEL = os.environ.get("EVAL_MODEL", COMPLETIONS_MODEL)

# This lab focuses on LLM-as-judge scoring; retrieval is mocked with a plain
# Cosmos query (see generate_response), so no embeddings client is needed —
# real embedding + vector search is covered in Labs 2D and 2E.
for var in ["COSMOS_ENDPOINT", "FOUNDRY_ENDPOINT"]:
    if not os.environ.get(var):
        raise RuntimeError(f"{var} environment variable is required.")

print(f"Cosmos Endpoint:   {ENDPOINT}")
print(f"Foundry Endpoint:  {FOUNDRY_ENDPOINT}")
print(f"Database:          {DB_NAME}")
print(f"Completions Model: {COMPLETIONS_MODEL}")
print(f"Eval Model:        {EVAL_MODEL}")

In [ ]:
from azure.cosmos import CosmosClient, PartitionKey
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

cred = DefaultAzureCredential()
cosmos_client = CosmosClient(url=ENDPOINT, credential=cred)
db = cosmos_client.get_database_client(DB_NAME)
container = db.get_container_client(CONT_NAME)
print(f"Connected to: {ENDPOINT}/{DB_NAME}/{CONT_NAME}")

# Chat completions: Foundry endpoint, Entra ID auth.
token_provider = get_bearer_token_provider(cred, "https://ai.azure.com/.default")
foundry_client = OpenAI(
    base_url=f"{FOUNDRY_ENDPOINT.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)
print("Foundry chat client initialized")

## Step 1: Create Evaluation Dataset (STUDENT EXERCISE)

Create a set of evaluation test cases with questions and ground truth answers.

**Expected output**: 3 evaluation examples with questions and ground truth.

In [ ]:
eval_dataset = [
    {"question": "What is Azure Cosmos DB?", "ground_truth": "Azure Cosmos DB is a globally distributed, multi-model database service from Microsoft."},
    {"question": "What types of indexes does Cosmos DB support?", "ground_truth": "Range, spatial, composite, vector, and full-text indexes."},
    {"question": "How do vector indexes differ from range indexes?", "ground_truth": "Vector indexes enable semantic similarity search on embeddings, while range indexes optimize numeric/string equality and ordering queries."}
]

print(f"Created {len(eval_dataset)} evaluation examples:")
for example in eval_dataset:
    print(f"  Q: {example['question']}")
    print(f"  Ground truth: {example['ground_truth']}\n")

## Step 2: Score RAG Outputs (STUDENT EXERCISE)

Use the LLM-as-judge pattern to score the relevance of RAG outputs against ground truth answers.

**Expected output**: Scores 1-5 for each evaluation example.

In [ ]:
import re

def generate_response(question: str) -> str:
    result = container.query_items(
        query="SELECT TOP 3 c.text FROM c WHERE c.partitionKey = 'rag'",
        partition_key="rag"
    )
    context = "\n\n".join(r.get("text", "") for r in list(result))
    system_prompt = f"You are a helpful assistant. Answer based on: {context}"
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": question}]
    completion = foundry_client.chat.completions.create(
        model=COMPLETIONS_MODEL, messages=messages, temperature=0.7, max_tokens=500
    )
    return completion.choices[0].message.content


print("Scoring answers using LLM-as-judge pattern...\n")
scores = []

for example in eval_dataset:
    answer = generate_response(example["question"])

    scoring_prompt = (
        f"Rate the answer's relevance to the question on a scale of 1 to 5.\n"
        f"Respond with ONLY a single digit (1, 2, 3, 4, or 5). No words, no punctuation, no explanation.\n\n"
        f"Question: {example['question']}\n"
        f"Answer: {answer}\n"
        f"Ground truth: {example['ground_truth']}\n\n"
        f"Score (single digit only):"
    )

    completion = foundry_client.chat.completions.create(
        model=EVAL_MODEL,
        messages=[{"role": "user", "content": scoring_prompt}],
        temperature=0.0,
        max_tokens=10
    )

    score_text = completion.choices[0].message.content.strip()
    match = re.search(r"[1-5]", score_text)
    if match:
        score = int(match.group(0))
        scores.append(score)
        print(f"Q: {example['question'][:40]}...")
        print(f"  Score: {score}/5")
    else:
        print(f"Q: {example['question'][:40]}...")
        print(f"  Score text could not be parsed: {score_text}")
        scores.append(0)
    print()

# Summarize results
print("=== Step 3: Summarize Results ===")
if scores:
    avg = sum(scores) / len(scores)
    print(f"Average relevance score: {avg:.2f}/5")
    if avg > 4:
        print("Recommendation: score >4 = good")
    elif avg >= 3:
        print("Recommendation: score 3-4 = needs improvement")
    else:
        print("Recommendation: score <3 = redesign RAG pipeline")

print("\n=== Lab Complete ===")
print("You have completed the evaluation exercise in Python. You:")
print("- Set up RAG pipeline with Cosmos DB and Azure OpenAI")
print("- Created an evaluation dataset")
print("- Scored RAG outputs using LLM-as-judge pattern")
print("- Summarized evaluation results")